# Building the point in time S&P 500 universe

Constructs a `(ticker, cik, start_date, end_date, source, left_censored)` spans table of
S&P 500 membership, 1996 to present, from two sources: Wikipedia's revision history (2008
onward, re-derivable from the primary source at any time) and a book-derived CSV (1996
through 2008, a frozen third party file). The reasoning behind every non-obvious decision
here, what was tried and rejected, each source's limitations, and the empirical cross-checks
between them are recorded in `notebooks/logs/universe_construction.md`. This notebook is the
working implementation; that file is the explanation.


In [1]:
# ssl/certifi: fix HTTPS cert verification on macOS python.org installs
# requests: makes the HTTP call so we control the headers
# pandas: parses HTML tables into DataFrames
import ssl, certifi, requests
import pandas as pd
from io import StringIO

# Point Python's default HTTPS context at certifi's CA bundle.
# The python.org installer doesn't hook into the macOS system keychain,
# so without this, HTTPS requests can fail with SSL: CERTIFICATE_VERIFY_FAILED.
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

# The page holding both tables we want: current constituents + historical changes log.
# %26 is the URL-encoded "&" in "S&P".
WIKI_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# Wikipedia returns 403 Forbidden to clients with a default/absent User-Agent.
# Presenting a real browser UA string gets us served normally.
HEADERS = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/124.0 Safari/537.36"}

# Fetch the page ourselves (rather than letting pandas fetch it) so our headers apply.
resp = requests.get(WIKI_URL, headers=HEADERS)

# requests does NOT raise on 4xx/5xx — it just returns a response with a bad status.
# This converts a bad status into an exception, so we fail loudly here at the fetch
# instead of confusingly later at the parse with an unhelpful "no tables found".
resp.raise_for_status()

# pandas 3.0 removed support for passing literal HTML strings — a bare string is
# interpreted as a file path. StringIO wraps the string in a file-like object,
# making it unambiguous that this is content, not a location.
tables = pd.read_html(StringIO(resp.text))

# Inspect what we actually got, rather than assuming the tables we want sit at
# fixed indices 0 and 1 — the page also contains sector navboxes and other tables,
# and their order can shift with any Wikipedia edit.
print(f"{len(tables)} tables found\n")
for i, t in enumerate(tables):
    print(f"--- table {i}: shape {t.shape} ---")
    print(list(t.columns))
    print()


3 tables found

--- table 0: shape (503, 8) ---
['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry', 'Headquarters Location', 'Date added', 'CIK', 'Founded']

--- table 1: shape (406, 6) ---
[('Effective Date', 'Effective Date'), ('Added', 'Ticker'), ('Added', 'Security'), ('Removed', 'Ticker'), ('Removed', 'Security'), ('Reason', 'Reason')]

--- table 2: shape (11, 2) ---
['vteS&P 500 companies', 'vteS&P 500 companies.1']



## Why not reconstruct from Wikipedia's changes log

The first approach tried was walking backward from today's constituent list through
Wikipedia's own "Selected changes" table, undoing each recorded change. That log turned out
to be incomplete (entire years, including 2001 to 2002 during the dot-com collapse, have
zero recorded changes), and reconstruction errors from an incomplete log accumulate
backward through time, corrupting the oldest dates first and worst. The per-year change
counts that established this, and the full reasoning, are in
`notebooks/logs/universe_construction.md`, under "Approach change: revision history
replaces changes log reconstruction."

The rest of this notebook uses the replacement: reading the page as it actually existed on
a given date, directly, through the MediaWiki revision API.


In [2]:
API_URL = "https://en.wikipedia.org/w/api.php"
PAGE_TITLE = "List of S&P 500 companies"

# Wikimedia asks API clients to identify themselves with a descriptive User-Agent
# including contact info — a browser UA string is fine for page scraping but
# considered impolite for API use, and can get you rate-limited.
API_HEADERS = {"User-Agent": "capm-portfolio/0.1 (research project; https://github.com/kc2998/capm-portfolio)"}

def revision_at(date_iso):
    """Return (revid, timestamp) of the last revision at or before date_iso."""
    params = {
        "action": "query", "prop": "revisions", "titles": PAGE_TITLE,
        "rvlimit": 1,
        "rvdir": "older",        # walk backward in time from rvstart
        "rvstart": date_iso,     # e.g. "2015-06-30T00:00:00Z"
        "rvprop": "ids|timestamp",
        "format": "json", "formatversion": 2,
    }
    r = requests.get(API_URL, params=params, headers=API_HEADERS, timeout=10)
    r.raise_for_status()
    revs = r.json()["query"]["pages"][0].get("revisions", [])
    return (revs[0]["revid"], revs[0]["timestamp"]) if revs else (None, None)

def revision_html(revid):
    """Fetch the rendered HTML of a specific revision."""
    params = {"action": "parse", "oldid": revid, "prop": "text",
              "format": "json", "formatversion": 2}
    r = requests.get(API_URL, params=params, headers=API_HEADERS, timeout=10)
    r.raise_for_status()
    return r.json()["parse"]["text"]

revid, ts = revision_at("2015-06-30T00:00:00Z")
print("revision:", revid, "timestamp:", ts)

snapshot_tables = pd.read_html(StringIO(revision_html(revid)))
print(f"\n{len(snapshot_tables)} tables in the 2015 revision\n")
for i, t in enumerate(snapshot_tables):
    print(f"--- table {i}: shape {t.shape} ---")
    print(list(t.columns))
    print()


revision: 669156533 timestamp: 2015-06-29T08:14:14Z

2 tables in the 2015 revision

--- table 0: shape (502, 8) ---
['Ticker symbol', 'Security', 'SEC filings', 'GICS Sector', 'GICS Sub Industry', 'Address of Headquarters', 'Date first added', 'CIK']

--- table 1: shape (104, 6) ---
[0, 1, 2, 3, 4, 5]



In [3]:
import time

def find_constituents(tabs):
    """Pick the table that looks like a constituents list: has a ticker-ish column.
    Matching on a column signature rather than a fixed index is what lets one parser
    work across a decade of Wikipedia reformatting."""
    for i, t in enumerate(tabs):
        cols = [str(c).lower() for c in t.columns]
        if any("symbol" in c or "ticker" in c for c in cols):
            return i, t
    return None, None

for d in ["2006-06-30", "2008-06-30", "2010-06-30", "2012-06-30",
          "2014-06-30", "2018-06-30", "2022-06-30"]:
    try:
        revid, ts = revision_at(f"{d}T00:00:00Z")
        if revid is None:
            print(f"{d}: no revision found")
            continue
        tabs = pd.read_html(StringIO(revision_html(revid)))
    except Exception as e:
        # Broad except is right *here*: the whole point is to survey which dates work
        # and which don't. We print the error type so a failure is still a data point.
        print(f"{d}: FAILED — {type(e).__name__}: {e}")
        continue

    i, t = find_constituents(tabs)
    if t is None:
        print(f"{d}: rev {revid} ({ts[:10]}) — {len(tabs)} tables, none ticker-like")
    else:
        print(f"{d}: rev {revid} ({ts[:10]}) — table {i}, {t.shape[0]} rows | {list(t.columns)}")
    time.sleep(0.5)


2006-06-30: FAILED — ValueError: No tables found
2008-06-30: rev 220621121 (2008-06-20) — table 0, 500 rows | ['Ticker symbol', 'Company', 'SEC filings', 'GICS Sector']
2010-06-30: rev 360899452 (2010-05-08) — table 0, 500 rows | ['Ticker symbol', 'Company', 'SEC filings', 'GICS Sector']
2012-06-30: rev 498154602 (2012-06-18) — table 0, 500 rows | ['Ticker symbol', 'Company', 'SEC filings', 'GICS Sector', 'Address of Headquarters']
2014-06-30: rev 614647412 (2014-06-27) — table 0, 501 rows | ['Ticker symbol', 'Security', 'SEC filings', 'GICS Sector', 'GICS Sub Industry', 'Address of Headquarters', 'Date first added', 'CIK']
2018-06-30: rev 848091676 (2018-06-29) — table 0, 505 rows | ['Ticker symbol', 'Security', 'SEC filings', 'GICS Sector', 'GICS Sub Industry', 'Location', 'Date first added[3][4]', 'CIK', 'Founded']
2022-06-30: rev 1095558369 (2022-06-29) — table 0, 503 rows | ['Symbol', 'Security', 'SEC filings', 'GICS Sector', 'GICS Sub-Industry', 'Headquarters Location', 'Date fir

In [4]:
import re

def normalize_ticker_punctuation(ticker):
    """Collapse the hyphen/period share-class notation to one form.

    Wikipedia itself is inconsistent across revisions about whether a
    multi-class ticker uses a period or a hyphen (BRK.B vs BRK-B), which
    otherwise makes build_spans see a punctuation change as an exit and a
    fresh entry for the same, unchanged security.
    """
    return re.sub(r"-([A-Za-z])$", r".\1", ticker)


def normalize_constituents(table):
    """Extract a clean (ticker, cik) frame from a raw constituents table.

    Column names drift across eras ('Ticker symbol' -> 'Symbol'; CIK absent
    before ~2014), so columns are matched by signature, the same principle
    find_constituents already uses to pick the table itself.
    """
    cols = {str(c).lower().strip(): c for c in table.columns}

    ticker_col = next((orig for lower, orig in cols.items()
                        if "symbol" in lower or "ticker" in lower), None)
    if ticker_col is None:
        raise ValueError("no ticker-like column found")

    cik_col = cols.get("cik")  # exact match — this column's name hasn't drifted

    return pd.DataFrame({
        "ticker": table[ticker_col].astype(str).str.strip().apply(normalize_ticker_punctuation),
        "cik": table[cik_col] if cik_col is not None else pd.NA,
    })


In [5]:
for d in ["2008-06-30", "2022-06-30"]:
    revid, _ = revision_at(f"{d}T00:00:00Z")
    tabs = pd.read_html(StringIO(revision_html(revid)))
    _, t = find_constituents(tabs)
    norm = normalize_constituents(t)
    print(d, norm.shape, "non-null CIK:", norm["cik"].notna().sum())
    display(norm.head())


2008-06-30 (500, 2) non-null CIK: 0


,ticker,cik
0,MMM,<NA>
1,ABT,<NA>
2,ANF,<NA>
3,ACE,<NA>
4,ADBE,<NA>


2022-06-30 (503, 2) non-null CIK: 503


,ticker,cik
0,MMM,66740
1,AOS,91142
2,ABT,1800
3,ABBV,1551152
4,ABMD,815094


In [6]:
def snapshot_at(date_iso):
    """Fetch and normalize the S&P 500 constituents table as of date_iso.

    Wraps revision_at, revision_html, find_constituents, and normalize_constituents
    into the single call the monthly loop needs. Returns None rather than
    raising when no usable table exists, either because the date predates the
    table's introduction on the page, or because the row count falls outside
    the 495-to-517 band the log established as plausible for this index —
    so a bad or vandalized revision gets skipped and counted, not silently
    trusted or allowed to crash a multi-hour loop.
    """
    revid, ts = revision_at(f"{date_iso}T00:00:00Z")
    if revid is None:
        return None

    try:
        tabs = pd.read_html(StringIO(revision_html(revid)))
    except ValueError:
        return None  # page fetched fine, but no tables on it yet — pre-2008 era

    _, table = find_constituents(tabs)
    if table is None:
        return None

    norm = normalize_constituents(table)
    if not (495 <= len(norm) <= 517):
        return None

    return norm, ts


for d in ["2006-06-30", "2008-06-30", "2022-06-30"]:
    result = snapshot_at(d)
    if result is None:
        print(d, "-> None")
    else:
        norm, ts = result
        print(d, "->", norm.shape, "revision timestamp", ts)


2006-06-30 -> None
2008-06-30 -> (500, 2) revision timestamp 2008-06-20T19:14:59Z
2022-06-30 -> (503, 2) revision timestamp 2022-06-29T02:08:14Z


## Monthly snapshots, 2008 to present

One snapshot per month end, the cadence already decided in the README (index membership
changes roughly twice a month, so a finer cadence would assert more precision than the
source has). Cached to `data/raw/wiki_snapshots.parquet` after the first run, this makes
roughly 440 requests and takes several minutes; every run after the first loads from disk
in under a second.


In [7]:
import os

SNAPSHOTS_CACHE = "../data/raw/wiki_snapshots.parquet"

if os.path.exists(SNAPSHOTS_CACHE):
    print("loading cached snapshots from", SNAPSHOTS_CACHE)
    cached = pd.read_parquet(SNAPSHOTS_CACHE)
    snapshots = {
        date: group.drop(columns="snapshot_date").reset_index(drop=True)
        for date, group in cached.groupby("snapshot_date")
    }
    skipped, errors = [], []
    print(len(snapshots), "snapshots loaded from cache")

else:
    month_ends = pd.date_range("2008-01-31", pd.Timestamp.today().normalize(), freq="ME")
    print(f"{len(month_ends)} months to fetch")

    snapshots = {}
    skipped = []
    errors = []

    for i, d in enumerate(month_ends):
        date_iso = d.strftime("%Y-%m-%d")
        try:
            result = snapshot_at(date_iso)
        except Exception as e:
            errors.append((date_iso, type(e).__name__, str(e)))
            continue

        if result is None:
            skipped.append(date_iso)
        else:
            norm, ts = result
            snapshots[date_iso] = norm

        if (i + 1) % 12 == 0:
            print(f"{i + 1}/{len(month_ends)} done — {len(skipped)} skipped, {len(errors)} errors")

        time.sleep(0.5)

    print(f"\n{len(snapshots)} snapshots retrieved, {len(skipped)} skipped, {len(errors)} errors")
    print("skipped:", skipped)
    print("errors:", errors)

    # Save point: one parquet file, so a kernel restart doesn't mean
    # redoing a 9-minute, ~440-request run against Wikipedia.
    all_snaps = pd.concat(
        [df.assign(snapshot_date=date) for date, df in snapshots.items()],
        ignore_index=True,
    )
    all_snaps["cik"] = all_snaps["cik"].astype("Int64")
    all_snaps.to_parquet(SNAPSHOTS_CACHE, index=False)
    print("cached to", SNAPSHOTS_CACHE)


222 months to fetch
12/222 done — 0 skipped, 0 errors
24/222 done — 0 skipped, 0 errors
36/222 done — 0 skipped, 0 errors
48/222 done — 0 skipped, 0 errors
60/222 done — 0 skipped, 0 errors
72/222 done — 0 skipped, 0 errors
84/222 done — 0 skipped, 0 errors
96/222 done — 0 skipped, 0 errors
108/222 done — 0 skipped, 0 errors
120/222 done — 0 skipped, 0 errors
132/222 done — 0 skipped, 0 errors
144/222 done — 0 skipped, 0 errors
156/222 done — 0 skipped, 0 errors
168/222 done — 0 skipped, 0 errors
180/222 done — 0 skipped, 0 errors
192/222 done — 0 skipped, 0 errors
204/222 done — 0 skipped, 0 errors
216/222 done — 0 skipped, 0 errors

222 snapshots retrieved, 0 skipped, 0 errors
skipped: []
errors: []
cached to ../data/raw/wiki_snapshots.parquet


## Snapshots to spans

Diffs consecutive monthly snapshots into one row per membership interval, the point in time
representation chosen over a wide membership matrix (see "Data model: spans table" in the
log).


In [8]:
def build_spans(snapshots):
    """Diff consecutive monthly snapshots into (ticker, start_date, end_date) spans.

    A ticker appearing in month N but not N-1 opens a span at month N; one
    disappearing closes the span at month N-1, the last date it was confirmed
    present. Boundary precision is therefore limited to the snapshot interval,
    matching what the log already established: exact to the month, not the day.
    """
    dates = sorted(snapshots)
    open_spans = {}   # ticker -> start_date
    closed = []       # list of (ticker, start_date, end_date)

    prev_date = None
    prev_tickers = set()

    for date in dates:
        curr_tickers = set(snapshots[date]["ticker"])

        entered = curr_tickers - prev_tickers
        exited = prev_tickers - curr_tickers

        for ticker in entered:
            open_spans[ticker] = date

        for ticker in exited:
            start = open_spans.pop(ticker)
            closed.append((ticker, start, prev_date))

        prev_tickers = curr_tickers
        prev_date = date

    # Still present at the last snapshot: no exit observed yet, so no end date.
    for ticker, start in open_spans.items():
        closed.append((ticker, start, None))

    return pd.DataFrame(closed, columns=["ticker", "start_date", "end_date"])

spans = build_spans(snapshots)
print(spans.shape)
spans.head()


(995, 3)


,ticker,start_date,end_date
0,CZN,2008-01-31,2008-07-31
1,OMX,2008-01-31,2008-07-31
2,IACI,2008-01-31,2008-07-31
3,BSC,2008-01-31,2008-07-31
4,CC,2008-01-31,2008-07-31


In [9]:
check_date = "2022-06-30"

from_spans = set(spans[
    (spans["start_date"] <= check_date)
    & (spans["end_date"].isna() | (spans["end_date"] >= check_date))
]["ticker"])

from_snapshot = set(snapshots[check_date]["ticker"])

print("match:", from_spans == from_snapshot)
print("in spans not snapshot:", from_spans - from_snapshot)
print("in snapshot not spans:", from_snapshot - from_spans)


match: True
in spans not snapshot: set()
in snapshot not spans: set()


## Attaching CIK

CIK (SEC's stable filer identifier, the EDGAR join key for fundamentals and insider data
later) is absent from Wikipedia's table before roughly 2014. Takes the latest non-null CIK
observed for each ticker across all snapshots, rather than whatever was present when the
span opened, so a span that opened before 2014 doesn't lose CIK it acquires later in the
same continuous membership.


In [10]:
def attach_cik(spans, snapshots):
    """Attach the latest known CIK for each ticker to its span(s).

    CIK is absent from Wikipedia's table before ~2014 (per the log), so a span
    opened in, say, 2009 has no CIK at its start date even if the same company
    is still a member after 2014, once the column exists. Taking the latest
    non-null CIK observed anywhere in the ticker's history, rather than the
    value at span start, avoids losing that information.
    """
    all_rows = pd.concat(snapshots.values(), ignore_index=True)
    latest_cik = (
        all_rows.dropna(subset=["cik"])
        .groupby("ticker")["cik"]
        .last()
    )
    spans = spans.copy()
    spans["cik"] = spans["ticker"].map(latest_cik)
    return spans

spans_with_cik = attach_cik(spans, snapshots)
print(spans_with_cik["cik"].notna().sum(), "of", len(spans_with_cik), "spans have a CIK")
spans_with_cik.head()


836 of 995 spans have a CIK


,ticker,start_date,end_date,cik
0,CZN,2008-01-31,2008-07-31,NaN
1,OMX,2008-01-31,2008-07-31,NaN
2,IACI,2008-01-31,2008-07-31,NaN
3,BSC,2008-01-31,2008-07-31,NaN
4,CC,2008-01-31,2008-07-31,NaN


In [11]:
missing_cik = spans_with_cik[spans_with_cik["cik"].isna()]
still_open = missing_cik["end_date"].isna().sum()

print(f"{len(missing_cik)} spans missing CIK, {still_open} of them still open (no end_date)")
print("latest end_date among missing-CIK spans:", missing_cik["end_date"].max())


159 spans missing CIK, 0 of them still open (no end_date)
latest end_date among missing-CIK spans: 2014-04-30


In [12]:
def flag_left_censored(spans, first_observed_date):
    """Mark spans whose start_date is the first date this source can see.

    A span opening on the very first observed snapshot doesn't mean the
    company joined the index that day, only that it was already a member
    when observation began. The true join date predates the data and is
    unrecoverable from this source (confirmed concretely for BCO in the
    book era, see the log). This matters for anything using tenure length
    as a signal; it does not affect membership correctness itself.
    """
    spans = spans.copy()
    spans["left_censored"] = spans["start_date"] == first_observed_date
    return spans

spans_with_cik = flag_left_censored(spans_with_cik, "2008-01-31")
print(spans_with_cik["left_censored"].sum(), "of", len(spans_with_cik), "wiki-era spans are left-censored")


500 of 995 wiki-era spans are left-censored


## Extending before 2008: the book CSV

Wikipedia's constituent table doesn't exist before 2008. A day-level CSV derived from
Andreas Clenow's *Trading Evolved* companion data (via Norgate) covers 1996 onward, but its
ticker labels can't be trusted at face value: still-active companies are relabeled with
their present-day ticker retroactively (confirmed concretely via `PCLN`/`BKNG`), and
delisted or ticker-recycled companies carry a `BASE-YYYYMM` disambiguating suffix that must
not be stripped, unlike what the source repository's own cleaning script does.

A 44-quarter empirical cross-check against the wiki source, and CIK-based reconciliation of
the residual mismatches, is recorded in full in the log, along with why the entire 2008 to
2019 overlap is excluded rather than blended with the Wikipedia era. Only the pre-2008
portion is used here.


In [13]:
def load_book_snapshots(path):
    """Load the book CSV's daily rows into the same {date: DataFrame} shape
    used for the Wikipedia snapshots, so build_spans works on either source
    unmodified.

    Every row in this file is already a full snapshot for one calendar day
    (though not every calendar day has a row — a new row appears only when
    something is worth recording), so there's no reason to resample it the
    way the Wikipedia loop's monthly cadence was forced by network cost.

    Tickers are kept exactly as the file reports them, including the
    BASE-YYYYMM suffix used when a ticker has since been recycled by a
    different company, per the finding in universe_construction.md.
    Stripping it, the way the sp500 tool's own cleaning script does, would
    merge two distinct companies' history under one symbol.
    """
    df = pd.read_csv(path)
    book_snapshots = {}
    for row in df.itertuples(index=False):
        tickers = row.tickers.split(",")
        book_snapshots[row.date] = pd.DataFrame({
            "ticker": tickers,
            "cik": pd.NA,
        })
    return book_snapshots

book_snapshots = load_book_snapshots(
    "../data/raw/S&P 500 Historical Components & Changes.csv"
)
print(len(book_snapshots), "daily snapshots loaded")
print(min(book_snapshots), "to", max(book_snapshots))


2595 daily snapshots loaded
1996-01-02 to 2019-01-11


In [14]:
book_snapshots_pre2008 = {
    date: df for date, df in book_snapshots.items() if date < "2008-01-31"
}

print(len(book_snapshots_pre2008), "snapshots before 2008-01-31")
print(min(book_snapshots_pre2008), "to", max(book_snapshots_pre2008))


1369 snapshots before 2008-01-31
1996-01-02 to 2008-01-30


In [15]:
book_spans = build_spans(book_snapshots_pre2008)
print(book_spans.shape)
book_spans.head()


(854, 3)


,ticker,start_date,end_date
0,BCO,1996-01-02,1996-01-12
1,CCB-199602,1996-01-02,1996-02-01
2,HDLM-201301,1996-01-02,1996-03-05
3,FBO-199603,1996-01-02,1996-03-12
4,CYR-199606,1996-01-02,1996-03-25


In [16]:
book_spans = flag_left_censored(book_spans, "1996-01-02")
print(book_spans["left_censored"].sum(), "of", len(book_spans), "spans are left-censored")


499 of 854 spans are left-censored


## CIK for the pre-2008 era

The book CSV has no CIK column at all. SEC's free bulk registry (`company_tickers.json`)
resolves it for any ticker still in use by an active filer today, which is most bare
tickers here, since the source itself already labels active companies with their present
day ticker. It cannot resolve a suffixed (`BASE-YYYYMM`) ticker, since that company is by
definition no longer an active filer; that non-match is expected and explainable, not a
defect to chase.


In [17]:
def fetch_sec_ticker_cik():
    """Fetch SEC's bulk ticker-to-CIK mapping for all currently registered filers.

    Free, no auth, roughly 10,000 entries. This only covers today's active
    registrants under today's current ticker, so it resolves a still-active
    company's present-day symbol (the book CSV's own convention for such
    companies, per the BKNG/PCLN finding) but can never resolve a ticker a
    company no longer trades under.
    """
    url = "https://www.sec.gov/files/company_tickers.json"
    headers = {"User-Agent": "capm-portfolio-research kevin (contact: hongxianl957@gmail.com)"}
    r = requests.get(url, headers=headers, timeout=10)
    r.raise_for_status()
    data = r.json()
    return {v["ticker"]: v["cik_str"] for v in data.values()}

SEC_CACHE = "../data/raw/sec_ticker_cik.parquet"

if os.path.exists(SEC_CACHE):
    print("loading cached SEC ticker-CIK map from", SEC_CACHE)
    sec_df = pd.read_parquet(SEC_CACHE)
    sec_ticker_to_cik = dict(zip(sec_df["ticker"], sec_df["cik"]))
else:
    sec_ticker_to_cik = fetch_sec_ticker_cik()
    pd.DataFrame({
        "ticker": list(sec_ticker_to_cik.keys()),
        "cik": list(sec_ticker_to_cik.values()),
    }).to_parquet(SEC_CACHE, index=False)
    print("cached to", SEC_CACHE)

print(len(sec_ticker_to_cik), "tickers in SEC's ticker-CIK map")


cached to ../data/raw/sec_ticker_cik.parquet
10432 tickers in SEC's ticker-CIK map


In [18]:
def base_ticker(t):
    """Strip the book CSV's BASE-YYYYMM disambiguation suffix, if present."""
    parts = t.split("-")
    return parts[0] if len(parts) == 2 and parts[1].isdigit() and len(parts[1]) == 6 else t

book_spans["cik"] = book_spans["ticker"].map(lambda t: sec_ticker_to_cik.get(base_ticker(t)))

match_rate = book_spans["cik"].notna().mean()
print(f"{book_spans['cik'].notna().sum()} of {len(book_spans)} book-era spans matched a CIK ({match_rate:.0%})")


432 of 854 book-era spans matched a CIK (51%)


## Ticker history: tracking symbol changes separately from membership

`universe_spans` answers "was this entity a member on date D." A second table,
`ticker_history`, answers a different question: "what symbol did this entity trade under on
date D," keyed by CIK rather than ticker so a rename doesn't look like two different
companies. Built with different confidence for each era: the Wikipedia era can be verified
directly, since its recorded ticker is already period-correct; the book era cannot be
verified the same way, since nothing as reliable as Wikipedia exists to check it against
before 2008. Full reasoning in the log.


In [19]:
def build_ticker_history_wiki(snapshots):
    """Build a (cik, ticker, start_date, end_date) history from the wiki snapshots.

    Unlike build_spans, which tracks membership, this tracks which ticker
    string a given CIK used over time. A CIK changing its associated ticker
    between two snapshots is a real, dated rename, verifiable because
    Wikipedia's ticker for each date is the one actually in use then. Rows
    with no CIK (pre-2014, per the log) can't be tracked this way; their
    tickers are still correct for membership purposes, just not linkable
    across a rename.
    """
    all_rows = pd.concat(
        [df.assign(snapshot_date=date) for date, df in snapshots.items()],
        ignore_index=True,
    )
    tracked = all_rows.dropna(subset=["cik"]).sort_values("snapshot_date")

    history = []
    for cik, group in tracked.groupby("cik"):
        current_ticker = None
        start = None
        for _, row in group.iterrows():
            if row["ticker"] != current_ticker:
                if current_ticker is not None:
                    history.append((cik, current_ticker, start, row["snapshot_date"]))
                current_ticker = row["ticker"]
                start = row["snapshot_date"]
        history.append((cik, current_ticker, start, None))

    return pd.DataFrame(history, columns=["cik", "ticker", "start_date", "end_date"])

wiki_ticker_history = build_ticker_history_wiki(snapshots)
wiki_ticker_history["source"] = "wikipedia_revision"
wiki_ticker_history["verified"] = True  # period-correct by construction, confirmed via PCLN/BKNG

renames_detected = (wiki_ticker_history.groupby("cik").size() > 1).sum()
print(renames_detected, "CIKs show more than one ticker across the wiki era")
wiki_ticker_history.head()


59 CIKs show more than one ticker across the wiki era


,cik,ticker,start_date,end_date,source,verified
0,1800,ABT,2014-05-31,NaN,wikipedia_revision,True
1,2488,AMD,2017-03-31,NaN,wikipedia_revision,True
2,2969,APD,2014-05-31,NaN,wikipedia_revision,True
3,4127,SWKS,2015-03-31,NaN,wikipedia_revision,True
4,4281,AA,2014-05-31,2016-11-30,wikipedia_revision,True


## Verifying book-era tickers against SEC's legal name history

SEC's `submissions` API (`data.sec.gov/submissions/CIK##########.json`) includes a free,
structured `formerNames` field, dated legal name history per company. It doesn't record
ticker history directly, but a ticker rename is almost always paired with a rebrand, so it
serves as a proxy: a CIK with no former name on record is unlikely to have changed ticker
either. This turns an unbounded, unverifiable risk (any bare book-era ticker might secretly
carry the same retroactive-relabeling problem confirmed for `BKNG`/`PCLN`, just undetected
before 2008) into a short, named list worth a manual look, rather than either trusting or
distrusting every bare ticker uniformly.

Suffixed tickers are not at risk here at all: they belong to companies no longer active by
the book file's generation date, and the retroactive-relabeling problem only affects
companies still active enough to have a "current" name to retroactively apply.


In [20]:
def fetch_former_names(cik):
    """Fetch a company's legal name history from SEC's submissions API.

    Returns a list of (name, from_date, to_date) tuples, empty if the
    company has never changed its legal name on record. Proxy signal only,
    not proof: a ticker can change without a legal renaming (a symbol
    conflict, an exchange switch), which this would miss.
    """
    url = f"https://data.sec.gov/submissions/CIK{int(cik):010d}.json"
    headers = {"User-Agent": "capm-portfolio-research kevin (contact: hongxianl957@gmail.com)"}
    r = requests.get(url, headers=headers, timeout=10)
    r.raise_for_status()
    data = r.json()
    return [(fn["name"], fn["from"][:10], fn["to"][:10]) for fn in data.get("formerNames", [])]


FORMER_NAMES_CACHE = "../data/raw/former_names.parquet"

if os.path.exists(FORMER_NAMES_CACHE):
    print("loading cached former names from", FORMER_NAMES_CACHE)
    former_names = pd.read_parquet(FORMER_NAMES_CACHE)
else:
    bare_ciks = book_spans.loc[
        book_spans["ticker"] == book_spans["ticker"].map(base_ticker), "cik"
    ].dropna().unique()
    print(len(bare_ciks), "distinct CIKs to check")

    rows = []
    for i, cik in enumerate(bare_ciks):
        try:
            for name, frm, to in fetch_former_names(cik):
                rows.append({"cik": cik, "former_name": name, "from_date": frm, "to_date": to})
        except Exception as e:
            print(cik, "failed:", type(e).__name__, e)
        if (i + 1) % 50 == 0:
            print(f"{i + 1}/{len(bare_ciks)} checked")
        time.sleep(0.15)  # SEC's fair-access limit is ~10 req/sec; comfortably under it

    former_names = pd.DataFrame(rows, columns=["cik", "former_name", "from_date", "to_date"])
    former_names.to_parquet(FORMER_NAMES_CACHE, index=False)
    print("cached to", FORMER_NAMES_CACHE)

print(len(former_names), "former-name records across", former_names["cik"].nunique(), "CIKs")


324 distinct CIKs to check
50/324 checked
100/324 checked
150/324 checked
200/324 checked
250/324 checked
300/324 checked
cached to ../data/raw/former_names.parquet
305 former-name records across 193 CIKs


In [21]:
def flag_book_ticker_verification(book_spans, former_names):
    """Mark each book-era span as verified or flagged for manual review.

    Suffixed tickers are verified by default; the retroactive-relabeling
    problem cannot apply to them (see above). A bare ticker is flagged if
    SEC records any former legal name for its CIK, on the reasoning given
    above. Deliberately over-inclusive (any former name at all, not a tight
    date-range check) since manual review of a flagged entry is cheap and
    missing one would not be. A bare ticker with no CIK match at all is
    flagged too: SEC's current registry not recognizing this exact symbol
    is itself evidence of a later rename or acquisition (the BHI/BHGE
    pattern from the cross-check), not proof the book's original label
    was safe.
    """
    book_spans = book_spans.copy()
    is_bare = book_spans["ticker"] == book_spans["ticker"].map(base_ticker)
    at_risk_ciks = set(former_names["cik"])
    no_cik_match = book_spans["cik"].isna()
    book_spans["verified"] = ~(is_bare & (book_spans["cik"].isin(at_risk_ciks) | no_cik_match))
    return book_spans

book_spans = flag_book_ticker_verification(book_spans, former_names)
print(f"{(~book_spans['verified']).sum()} of {len(book_spans)} book-era spans flagged for manual review")
book_spans.loc[~book_spans["verified"], ["ticker", "cik", "start_date", "end_date"]]


286 of 854 book-era spans flagged for manual review


,ticker,cik,start_date,end_date
0,BCO,78890.0,1996-01-02,1996-01-12
10,CAL,14707.0,1996-01-02,1996-07-19
16,YRCW,NaN,1996-01-02,1996-10-28
21,LUB,NaN,1996-01-02,1996-12-27
86,FL,NaN,1996-01-02,1998-12-31
...,...,...,...,...
848,AMT,1053507.0,2007-11-19,NaN
849,GME,1326380.0,2007-12-14,NaN
850,RRC,315852.0,2007-12-21,NaN
851,GHC,104889.0,2007-12-31,NaN


In [22]:
book_ticker_history = book_spans[["cik", "ticker", "start_date", "end_date", "verified"]].copy()
book_ticker_history["source"] = "clenow_norgate"

ticker_history = pd.concat([book_ticker_history, wiki_ticker_history], ignore_index=True)
ticker_history["cik"] = ticker_history["cik"].astype("Int64")

TICKER_HISTORY_PATH = "../data/processed/ticker_history.parquet"
os.makedirs("../data/processed", exist_ok=True)
ticker_history.to_parquet(TICKER_HISTORY_PATH, index=False)
print(ticker_history.shape, "saved to", TICKER_HISTORY_PATH)


(2809, 6) saved to ../data/processed/ticker_history.parquet


## Combining both eras, with boundary stitching

A straight concatenation would leave a company continuously in the index across 2008-01-31
as two disconnected rows, one ending oddly (open, because the book data was deliberately
cut off there, not because membership actually ended) and one starting fresh and
left-censored. Shared CIK lets these be recognized as one continuous membership and merged,
rather than read as a phantom exit and re-entry.


In [23]:
def combine_universe_spans(book_spans, spans_with_cik, boundary_date="2008-01-31", book_era_end="2008-01-30"):
    """Concatenate both eras' spans, stitching any membership that straddles
    the boundary where the two sources meet into a single row.

    A book-era span left open at the boundary (no exit observed, only
    because the book data stops there) and a wiki-era span opening exactly
    at the boundary, sharing a CIK, are one continuous membership recorded
    by two different sources, not two real events. Left as two adjacent
    rows when no shared CIK confirms it, most often because the book-era
    company has no CIK match at all (typically delisted before 2008,
    never reaching Wikipedia's coverage). A book-era span left open with
    no continuing wiki-era entry has its end_date capped at book_era_end,
    the last date the book source was actually observed. Leaving it null
    would claim the company is still active today, which only holds for
    the wiki era, where the most recent snapshot is close to today; the
    book era's coverage simply stops.
    """
    book_open = book_spans[book_spans["end_date"].isna() & book_spans["cik"].notna()]
    wiki_at_boundary = spans_with_cik[
        (spans_with_cik["start_date"] == boundary_date) & spans_with_cik["cik"].notna()
    ]
    stitch_ciks = set(book_open["cik"]) & set(wiki_at_boundary["cik"])
    print(f"{len(stitch_ciks)} memberships stitched across the 2008 boundary")

    stitched = []
    for cik in stitch_ciks:
        book_row = book_open[book_open["cik"] == cik].iloc[0]
        wiki_row = wiki_at_boundary[wiki_at_boundary["cik"] == cik].iloc[0]
        stitched.append({
            "ticker": wiki_row["ticker"],
            "cik": cik,
            "start_date": book_row["start_date"],
            "end_date": wiki_row["end_date"],
            "source": "clenow_norgate+wikipedia_revision",
            "left_censored": book_row["left_censored"],
        })
    stitched_df = pd.DataFrame(stitched)

    book_remaining = book_spans[~(
        book_spans["end_date"].isna() & book_spans["cik"].isin(stitch_ciks)
    )].copy()
    still_open = book_remaining["end_date"].isna()
    print(f"{still_open.sum()} book-era spans left open with no continuing wiki entry, "
          f"capped at {book_era_end} rather than left open-ended")
    book_remaining.loc[still_open, "end_date"] = book_era_end
    book_remaining = book_remaining.assign(source="clenow_norgate")

    wiki_remaining = spans_with_cik[~(
        (spans_with_cik["start_date"] == boundary_date) & spans_with_cik["cik"].isin(stitch_ciks)
    )].assign(source="wikipedia_revision")

    universe_spans = pd.concat([book_remaining, wiki_remaining, stitched_df], ignore_index=True)
    universe_spans["cik"] = universe_spans["cik"].astype("Int64")
    return universe_spans[["ticker", "cik", "start_date", "end_date", "source", "left_censored"]]


universe_spans = combine_universe_spans(book_spans, spans_with_cik)
print(universe_spans.shape)

os.makedirs("../data/processed", exist_ok=True)
UNIVERSE_PATH = "../data/processed/universe_spans.parquet"
universe_spans.to_parquet(UNIVERSE_PATH, index=False)
print("saved to", UNIVERSE_PATH)


276 memberships stitched across the 2008 boundary
224 book-era spans left open with no continuing wiki entry, capped at 2008-01-30 rather than left open-ended
(1569, 6)
saved to ../data/processed/universe_spans.parquet


In [24]:
def membership_on(universe_spans, date_iso):
    """Return the set of tickers that were S&P 500 members on date_iso."""
    active = universe_spans[
        (universe_spans["start_date"] <= date_iso)
        & (universe_spans["end_date"].isna() | (universe_spans["end_date"] >= date_iso))
    ]
    return set(active["ticker"])

for d in ["2000-01-03", "2008-01-31", "2015-06-30", "2022-06-30"]:
    print(d, "->", len(membership_on(universe_spans, d)), "members")


2000-01-03 -> 499 members
2008-01-31 -> 498 members
2015-06-30 -> 500 members
2022-06-30 -> 501 members


## Automated verification: checking flagged tickers against SEC filings

193 of the book era's spans are flagged for manual review (either no CIK match at all, or a
CIK with a legal name change on record). Rather than asking an agent to search the web and
assert an answer, not reproducible, and no way to audit a wrong guess beyond asking again,
each flagged, CIK-matched span is checked against that company's own historical SEC filings
directly: find the 10-K closest to the span's date, fetch its actual text, and search for a
literal ticker disclosure. Every result carries a citable filing and quoted sentence, or an
honest "not found," never a bare assertion. Full reasoning, including why an agent-scraped
answer would be a weaker source than the curated repositories already rejected elsewhere in
this project, is in the log.


In [25]:
def get_filing_history(cik):
    """Fetch a CIK's complete filing history, not just the recent ~1000 filings.

    The submissions endpoint's top-level filings.recent only covers a
    company's most recent filings; anything older is referenced under
    filings.files as separate paginated JSON files, which is where a
    company's 1990s-era 10-Ks actually live (confirmed for CAL/Caleres:
    its 1996 10-K405 only shows up in the paginated file, not filings.recent).
    """
    url = f"https://data.sec.gov/submissions/CIK{int(cik):010d}.json"
    headers = {"User-Agent": "capm-portfolio-research kevin (contact: hongxianl957@gmail.com)"}
    r = requests.get(url, headers=headers, timeout=10)
    r.raise_for_status()
    data = r.json()

    recent = data["filings"]["recent"]
    rows = list(zip(recent["form"], recent["filingDate"], recent["accessionNumber"]))

    for f in data["filings"].get("files", []):
        page_url = f"https://data.sec.gov/submissions/{f['name']}"
        pr = requests.get(page_url, headers=headers, timeout=10)
        pr.raise_for_status()
        page = pr.json()
        rows.extend(zip(page["form"], page["filingDate"], page["accessionNumber"]))

    return rows

def nearest_10k(history, target_date, max_gap_days=548):
    """Pick the 10-K-type filing closest to target_date.

    Returns None if the closest available filing is still more than
    max_gap_days (about 18 months) away, since a distant filing (like a
    2003 filing for a 1996-2000 span) isn't evidence about the span at
    all, and reporting it as a "found filing" would be misleading rather
    than informative.
    """
    candidates = [(f, d, a) for f, d, a in history if f.startswith("10-K")]
    if not candidates:
        return None
    target = pd.Timestamp(target_date)
    candidates.sort(key=lambda row: abs(pd.Timestamp(row[1]) - target))
    nearest = candidates[0]
    gap = abs(pd.Timestamp(nearest[1]) - target)
    if gap > pd.Timedelta(days=max_gap_days):
        return None
    return nearest



In [26]:
def fetch_filing_text(cik, accession):
    """Fetch a filing's complete submission text file.

    Old EDGAR filings aren't valid UTF-8; decoding as latin-1, which never
    raises on arbitrary bytes, avoids encoding errors on this older data.
    """
    url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession}.txt"
    headers = {"User-Agent": "capm-portfolio-research kevin (contact: hongxianl957@gmail.com)"}
    r = requests.get(url, headers=headers, timeout=15)
    r.raise_for_status()
    return r.content.decode("latin-1")


def extract_ticker_mentions(text):
    """Find candidate ticker mentions, tolerating quotes, colons, and paired tickers.

    Tickers are frequently quoted in older filings ('symbol "HP."') or
    listed factsheet-style ('Symbol: PMTC'), and a single sentence
    sometimes discloses two at once for a dual class or tracking stock
    structure ('ticker symbols "PZB" and "PZX"'). Only "symbol(s)" itself
    is matched case-insensitively; each captured ticker must still be
    genuinely all-caps in the source text, or ordinary lowercase words
    right after "symbol" get mistaken for tickers.
    """
    pattern = re.compile(
        r'.{0,60}(?i:symbols?)\s*:?\s*'
        r'["\']?([A-Z]{1,5})[.,]?["\']?'
        r'(?:\s*(?:and|,)\s*["\']?([A-Z]{1,5})[.,]?["\']?)?'
        r'.{0,40}'
    )
    matches = []
    for m in pattern.finditer(text):
        for ticker in m.groups():
            if ticker:
                matches.append((ticker, m.group(0).strip()))
    return matches




def find_ticker_in_filing(cik, target_date):
    """Find the ticker actually disclosed in a filing near target_date.

    Walks the CIK's complete filing history, picks the 10-K closest to
    target_date, fetches its full text, and searches for a ticker mention.
    Returns None if no 10-K exists near this date, or a dict carrying the
    filing's identity and matched evidence, so a failure is a reported gap
    rather than a guess.
    """
    try:
        history = get_filing_history(cik)
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"}

    filing = nearest_10k(history, target_date)
    if filing is None:
        return None

    form, filing_date, accession = filing
    try:
        text = fetch_filing_text(cik, accession)
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"}

    return {
        "form": form,
        "filing_date": filing_date,
        "accession": accession,
        "mentions": extract_ticker_mentions(text),
    }


In [27]:
sample = book_spans[(~book_spans["verified"]) & book_spans["cik"].notna()].head(5)

for _, row in sample.iterrows():
    result = find_ticker_in_filing(row["cik"], row["start_date"])
    print(f"--- {row['ticker']} (cik {int(row['cik'])}, span {row['start_date']} to {row['end_date']}) ---")
    if result is None:
        print("  no 10-K found near this date")
    elif "error" in result:
        print("  fetch failed:", result["error"])
    else:
        print(f"  nearest filing: {result['form']} filed {result['filing_date']} (accession {result['accession']})")
        if result["mentions"]:
            for ticker, context in result["mentions"][:3]:
                print(f'    candidate: {ticker}  |  "...{context}..."')
        else:
            print("    no ticker-pattern match found in filing text")
    time.sleep(0.3)
    print()


--- BCO (cik 78890, span 1996-01-02 to 1996-01-12) ---
  nearest filing: 10-K405 filed 1996-04-01 (accession 0000950117-96-000277)
    candidate: PZS  |  "...York Stock Exchange under the ticker symbols "PZS" and "PZM", respectively...."
    candidate: PZM  |  "...York Stock Exchange under the ticker symbols "PZS" and "PZM", respectively...."
    candidate: PZB  |  "...tocks trade on the New York Stock Exchange under the ticker symbols "PZB"
and "PZX", respectively...."

--- CAL (cik 14707, span 1996-01-02 to 1996-07-19) ---
  nearest filing: 10-K405 filed 1996-04-19 (accession 0000014707-96-000004)
    candidate: BG  |  "...(symbol BG). There were approximately 6,000 shareh..."
    candidate: BG  |  "...Stock Exchange (ticker symbol BG)...."

--- HP (cik 46765, span 1996-01-02 to 1999-12-28) ---
  nearest filing: 10-K filed 1995-12-22 (accession 0000950134-95-003420)
    candidate: HP  |  "...with the ticker symbol "HP." The newspaper abbreviation most commonl..."

--- SCI (cik 89089,

In [28]:
FILING_VERIFICATION_CACHE = "../data/raw/filing_verification.parquet"

if os.path.exists(FILING_VERIFICATION_CACHE):
    print("loading cached filing verification from", FILING_VERIFICATION_CACHE)
    filing_verification = pd.read_parquet(FILING_VERIFICATION_CACHE)
else:
    to_check = book_spans[(~book_spans["verified"]) & book_spans["cik"].notna()]
    print(len(to_check), "flagged, CIK-matched spans to check")

    rows = []
    for i, (_, row) in enumerate(to_check.iterrows()):
        result = find_ticker_in_filing(row["cik"], row["start_date"])
        base = {"ticker": row["ticker"], "cik": row["cik"],
                "start_date": row["start_date"], "end_date": row["end_date"]}

        if result is None:
            rows.append({**base, "status": "no_filing_found", "candidate": None,
                         "evidence": None, "accession": None})
        elif "error" in result:
            rows.append({**base, "status": "fetch_error", "candidate": None,
                         "evidence": result["error"], "accession": None})
        elif not result["mentions"]:
            rows.append({**base, "status": "no_pattern_match", "candidate": None,
                         "evidence": None, "accession": result["accession"]})
        else:
            candidates = {t for t, _ in result["mentions"]}
            if row["ticker"] in candidates:
                status = "confirmed_match"
            elif len(candidates) == 1:
                status = "confirmed_mismatch"
            else:
                status = "confirmed_mismatch_ambiguous"
            rows.append({**base, "status": status, "candidate": ", ".join(sorted(candidates)),
                         "evidence": result["mentions"][0][1], "accession": result["accession"]})

        if (i + 1) % 25 == 0:
            print(f"{i + 1}/{len(to_check)} checked")
        time.sleep(0.3)

    filing_verification = pd.DataFrame(rows)
    filing_verification.to_parquet(FILING_VERIFICATION_CACHE, index=False)
    print("cached to", FILING_VERIFICATION_CACHE)

print(filing_verification["status"].value_counts())


193 flagged, CIK-matched spans to check
25/193 checked
50/193 checked
75/193 checked
100/193 checked
125/193 checked
150/193 checked
175/193 checked
cached to ../data/raw/filing_verification.parquet
status
no_pattern_match                82
confirmed_match                 61
no_filing_found                 29
confirmed_mismatch              18
confirmed_mismatch_ambiguous     3
Name: count, dtype: int64


## Applying verified results back into ticker_history

82 of the 193 flagged spans now have real evidence one way or the other; the other 111 stay
an honest, explained residual (no filing close enough in time, or a filing that simply
doesn't restate its own ticker). Confirmed matches get a stronger, citation-backed verified
status; confirmed mismatches get corrected, with the original book ticker and the filing
citation kept alongside rather than silently overwritten; ambiguous mismatches (multiple
tracking-stock or dual class tickers found, like `BCO`/Pittston) are left for a person to
resolve, not auto-corrected.


In [29]:
def apply_filing_verification(ticker_history, filing_verification):
    """Apply the SEC filing verification results back into ticker_history.

    confirmed_match spans get a stronger verified status backed by an
    actual citation, not just the absence of a red flag. confirmed_mismatch
    spans get their ticker corrected, with the original book label and the
    filing citation kept alongside rather than silently overwritten, so the
    correction stays auditable. confirmed_mismatch_ambiguous spans are left
    untouched: multiple candidate tickers (a dual class or tracking stock
    split, like BCO/Pittston) need a person to pick the right one, not an
    automatic choice. Everything else (no_filing_found, no_pattern_match,
    fetch_error) is left exactly as it was, an honestly unresolved gap,
    not silently dropped.
    """
    ticker_history = ticker_history.copy()
    ticker_history["original_ticker"] = pd.NA
    ticker_history["evidence"] = pd.NA

    for _, row in filing_verification.iterrows():
        mask = (
            (ticker_history["source"] == "clenow_norgate")
            & (ticker_history["cik"] == row["cik"])
            & (ticker_history["start_date"] == row["start_date"])
        )
        if row["status"] == "confirmed_match":
            ticker_history.loc[mask, "verified"] = True
            ticker_history.loc[mask, "evidence"] = f"{row['accession']}: {row['evidence']}"
        elif row["status"] == "confirmed_mismatch":
            ticker_history.loc[mask, "original_ticker"] = ticker_history.loc[mask, "ticker"]
            ticker_history.loc[mask, "ticker"] = row["candidate"]
            ticker_history.loc[mask, "verified"] = True
            ticker_history.loc[mask, "evidence"] = f"{row['accession']}: {row['evidence']}"
        # confirmed_mismatch_ambiguous, no_filing_found, no_pattern_match,
        # fetch_error: left untouched, still verified = False, no evidence recorded.

    return ticker_history


ticker_history = apply_filing_verification(ticker_history, filing_verification)

print(ticker_history["verified"].value_counts())
print(ticker_history["original_ticker"].notna().sum(), "tickers corrected with a citation")

TICKER_HISTORY_PATH = "../data/processed/ticker_history.parquet"
ticker_history.to_parquet(TICKER_HISTORY_PATH, index=False)
print("re-saved to", TICKER_HISTORY_PATH)


verified
True     2602
False     207
Name: count, dtype: int64
20 tickers corrected with a citation
re-saved to ../data/processed/ticker_history.parquet
